# Agentic RAG Demo
This notebook demonstrates the Adaptive RAG, Corrective RAG (CRAG), and Self-RAG pipelines working together.

In [1]:
import sys
from pathlib import Path

# Ensure project root is in path so 'src' imports work
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from main import agentic_rag_app


c:\Users\amanm\Desktop\learning\developer-chat-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Index 'agenticrag' already exists


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11686.77it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 1. Direct LLM Route
For general queries, the Adaptive Router bypasses retrieval and uses the LLM directly.

In [2]:
query = "what is 2+2?"

initial_state = {
    "query": query,
    "original_query": query,
    "namespace": "default_namespace",
    "documents": [],
    "final_context_strips": [],
    "needs_web_search": False,
    "answer": "",
    "retries": 0,
    "route": ""
}

for chunk in agentic_rag_app.stream(initial_state):
    for node, update in chunk.items():
        print(f"\n===> Update from node [{node}]:")
        if 'route' in update:
            print(f"Route chosen: {update['route']}")
        if 'answer' in update and update['answer']:
            print(f"Answer: \n{update['answer']}\n")


2026-03-28 23:28:42 - src.adaptive_router - INFO - Query 'what is 2+2?' classified to route: 'llm_direct'

===> Update from node [router]:
Route chosen: llm_direct

===> Update from node [generate]:
Answer: 
4



## 2. Vectorstore Route (CRAG + Self-RAG)
For domain specific queries, the Adaptive Router will choose vectorstore. Then CRAG assesses documents and Self-RAG checks hallucinations/quality.

In [3]:
query = "Tell me about the cross-modal architecture of RAG-Anything."

initial_state_2 = {
    "query": query,
    "original_query": query,
    "namespace": "default_namespace",
    "documents": [],
    "final_context_strips": [],
    "needs_web_search": False,
    "answer": "",
    "retries": 0,
    "route": ""
}

for chunk in agentic_rag_app.stream(initial_state_2):
    for node, update in chunk.items():
        print(f"\n===> Update from node [{node}]:")
        if 'documents' in update:
            print(f"Retrieved {len(update['documents'])} documents.")
        if 'final_context_strips' in update:
            print(f"Relevant context strips: {len(update['final_context_strips'])}")
        if 'needs_web_search' in update:
            print(f"Needs web search fallback: {update['needs_web_search']}")
        if 'answer' in update and update['answer']:
            print(f"Answer: \n{update['answer']}\n")


2026-03-28 23:28:47 - src.adaptive_router - INFO - Query 'Tell me about the cross-modal architecture of RAG-Anything.' classified to route: 'vectorstore'

===> Update from node [router]:
2026-03-28 23:28:47 - src.workflows.crag - INFO - Using Serverless Pinecone Reranking Model
2026-03-28 23:28:52 - src.retrieval.reranker - INFO - Reranker returned 5 hits (top_n=5)

===> Update from node [retrieve]:
Retrieved 5 documents.

===> Update from node [evaluate]:
Relevant context strips: 18
Needs web search fallback: False
2026-03-28 23:29:55 - src.workflows.self_rag - INFO - Hallucination Score: no

===> Update from node [generate]:
2026-03-28 23:30:14 - src.workflows.self_rag - INFO - Hallucination Score: yes
2026-03-28 23:30:18 - src.workflows.self_rag - INFO - Answer Quality Score: yes

===> Update from node [generate]:
Answer: 
**Cross‑Modal Architecture of RAG‑Anything**

RAG‑Anything extends the classic Retrieval‑Augmented Generation (RAG) pipeline to handle *any* modality—text, images

## 3. Web Search Route (Fallback)
For current events, the adaptive router or fallback logic triggers a web search.

In [4]:
query = "What is the weather like of Delhi?"

initial_state_3 = {
    "query": query,
    "original_query": query,
    "namespace": "default_namespace",
    "documents": [],
    "final_context_strips": [],
    "needs_web_search": False,
    "answer": "",
    "retries": 0,
    "route": ""
}

for chunk in agentic_rag_app.stream(initial_state_3):
    for node, update in chunk.items():
        print(f"\n===> Update from node [{node}]:")
        if 'route' in update:
            print(f"Route chosen: {update['route']}")
        if 'final_context_strips' in update:
            print(f"Context obtained.")
        if 'answer' in update and update['answer']:
            print(f"Answer: \n{update['answer']}\n")


2026-03-28 23:31:09 - src.adaptive_router - INFO - Query 'What is the weather like of Delhi?' classified to route: 'web_search'

===> Update from node [router]:
Route chosen: web_search

===> Update from node [web_search]:
Context obtained.
2026-03-28 23:31:11 - src.workflows.self_rag - INFO - Answer Quality Score: yes

===> Update from node [generate]:
Answer: 
**Delhi’s climate**

- **General pattern** – Delhi has a humid subtropical climate (Köppen Cwa) that borders on a hot semi‑arid climate (BSh).  
- **Summers** – Very hot, often above 35 °C, with frequent thunderstorms known locally as *andhi*.  
- **Winters** – Mild to cool, with dense fog that can reduce visibility.  
- **Rainfall** – Monsoon season (June–September) brings the bulk of the annual precipitation, while the rest of the year is relatively dry.

**Current weather snapshot (as of the latest report)**  

| Parameter | Value |
|-----------|-------|
| Condition | Mist |
| Temperature | 31 °C |
| Humidity | 29 % |
| Wind